## Exploração e Preparação dos Dados

O objetivo deste notebook é explorar os dados disponibilizados pela API da StatsBomb e identificar um jogo que contenha informação suficiente para desenvolver uma análise baseada em Event Data e StatsBomb 360 Data.

Ao longo desta fase serão realizadas as seguintes etapas:

- exploração dos endpoints disponíveis na API;
- identificação de uma competição e jogo adequados para análise;
- validação da disponibilidade de eventos, lineups e dados 360;
- exploração da estrutura dos datasets;
- criação de datasets tratados para utilização posterior;
- definição de um modelo analítico que servirá de base para a fase de feature engineering e cisualização.

O output deste notebook será um conjunto de tabelas tratadas que servirão de input para o desenvolvimento de métricas e para a construção do relatório final.

****

In [131]:
# instalar primeiras dependências

!pip install statsbombpy pandas

In [132]:
# importações

import pandas as pd
from statsbombpy import sb

In [133]:
# antes de começar a aceder aos endpoints preciso de saber quais existem

from statsbombpy import sb
[x for x in dir(sb) if not x.startswith("_")]

['DEFAULT_CREDS',
 'MAX_CONCURRENCY',
 'Pool',
 'Union',
 'api_client',
 'competition_events',
 'competition_frames',
 'competitions',
 'events',
 'filter_and_group_events',
 'frames',
 'lineups',
 'matches',
 'merge_events_and_frames',
 'partial',
 'pd',
 'player_match_stats',
 'player_season_stats',
 'public',
 'reduce_events',
 'team_season_stats']

A ideia será por começar a explorar os datasets que vou usar, de forma a criar posteriormente um starschema na parte da visualização.
Vou começar por verificar competitions (que deverá dar também info de seasons), matches, lineups, events, frames.

****

## Seleção da Competição e do Jogo

Antes de iniciar a exploração dos dados é necessário identificar uma competição e uma partida que disponibilizem informação suficiente para o desenvolvimento do projeto.

Como o objetivo passa por combinar Event Data e StatsBomb 360 Data, será dada prioridade a jogos que possuam:
- dados de eventos
- dados de lineups
- dados de freeze frames (360)
- informação contextual suficiente para o desenvolvimento de métricas espaciais

<br>

## Competitions

In [134]:
# consulta ao dataset competitions para perceber competições disponíveis

competitions = sb.competitions()
competitions.head()

# (já traz season_ids mas ainda não sei a que seasons reais correspondem)

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,None,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882


In [135]:
# pesquisa rápida a todos as colunas existentes no dataset de competitions

competitions.columns

Index(['competition_id', 'season_id', 'country_name', 'competition_name',
       'competition_gender', 'competition_youth', 'competition_international',
       'season_name', 'match_updated', 'match_updated_360',
       'match_available_360', 'match_available'],
      dtype='object')

In [136]:
# este código dá-me um overview mais resumida do que para já quero (ids e nomes de competição e época, para poder interpretar)

competitions[
    [
        "competition_id",
        "season_id",
        "competition_name",
        "season_name"
    ]
].sort_values(
    by=["competition_id", "season_id"],
    ascending=[True, False]
).head(5)

,competition_id,season_id,competition_name,season_name
69,2,44,Premier League,2003/2004
68,2,27,Premier League,2015/2016
61,7,235,Ligue 1,2022/2023
62,7,108,Ligue 1,2021/2022
63,7,27,Ligue 1,2015/2016


In [137]:
# vou procurar um jogo de Campeonato do Mundo para perceber se é uma das competições que tem eventos e 360
# (se match_updated_360, match_available_360 e match_available não for 'None', assume-se como uma boa possibilidade de escolha)

world_cup = (
    competitions[
        competitions["competition_name"].str.contains("World Cup", na=False)
    ]
    .dropna(
        subset=["match_updated_360", "match_available_360", "match_available"]
    )
    .sort_values(
        by="season_name",
        ascending=False
    )
    .reset_index(drop=True)
)

world_cup

# season_id '106' + competition_id '43' pode ser uma boa possibilidade de escolha de jogo com a informação pretendida

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141
1,43,106,International,FIFA World Cup,male,False,True,2022,2026-05-04T01:48:57.914346,2026-05-04T01:53:40.309717,2026-05-04T01:53:40.309717,2026-05-04T01:48:57.914346


****

## Matches

Consulta ao dataset matches para perceber informações disponíveis:

- matches = sb.matches()
- matches.head()

Para este código tentei seguir o mesmo método de competitions, mas são requeridos os argumentos 'competition_id' and 'season_id'

In [138]:
# como já tenho os argumento em cima, é só adicionar ao código

matches = (
    sb.matches(
        competition_id=43,
        season_id=106
    )
    .sort_values(
        by="match_date",
        ascending=False
    )
    .reset_index(drop=True)
)

matches.head()

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3869685,2022-12-18,17:00:00.000,International - FIFA World Cup,2022,Argentina,France,3,3,available,...,2024-12-16T10:21:13.710934,7,Final,Lusail Stadium,Szymon Marciniak,Lionel Sebastián Scaloni,Didier Deschamps,1.1.0,2,2
1,3869684,2022-12-17,17:00:00.000,International - FIFA World Cup,2022,Croatia,Morocco,2,1,available,...,2024-02-13T13:44:03.865296,7,3rd Place Final,Sheikh Khalifa International Stadium,Abdulrahman Ibrahim Al Jassim,Zlatko Dalić,Walid Regragui,1.1.0,2,2
2,3869552,2022-12-14,21:00:00.000,International - FIFA World Cup,2022,France,Morocco,2,0,available,...,2024-02-06T16:45:42.775801,6,Semi-finals,Al Bayt Stadium,César Arturo Ramos Palazuelos,Didier Deschamps,Hoalid Regragui,1.1.0,2,2
3,3869519,2022-12-13,21:00:00.000,International - FIFA World Cup,2022,Argentina,Croatia,3,0,available,...,2023-04-26T22:32:37.808359,6,Semi-finals,Lusail Stadium,Daniele Orsato,Lionel Sebastián Scaloni,Zlatko Dalić,1.1.0,2,2
4,3869486,2022-12-10,17:00:00.000,International - FIFA World Cup,2022,Morocco,Portugal,1,0,available,...,2024-02-06T16:39:05.200810,5,Quarter-finals,Al Thumama Stadium,Facundo Tello Figueroa,Hoalid Regragui,Fernando Manuel Fernandes da Costa Santos,1.1.0,2,2


In [139]:
# pesquisa rápida a todos as colunas existentes no dataset de matches

matches.columns

Index(['match_id', 'match_date', 'kick_off', 'competition', 'season',
       'home_team', 'away_team', 'home_score', 'away_score', 'match_status',
       'match_status_360', 'last_updated', 'last_updated_360', 'match_week',
       'competition_stage', 'stadium', 'referee', 'home_managers',
       'away_managers', 'data_version', 'shot_fidelity_version',
       'xy_fidelity_version'],
      dtype='object')

In [140]:
# tal como em cima, vou tentar escolher um jogo que tem eventos e dados 360
# vou assumir que se match_status_360 não for 'None', o jogo é válido

matches = (
    sb.matches(
        competition_id=43,
        season_id=106
    )
    .dropna(
        subset=["match_status_360"]
    )
    .sort_values(
        by="match_date",
        ascending=False
    )
    .reset_index(drop=True)
)

matches.head()

# a escolha recairá então sob o jogo da final, Argentina x France, match_id = 3869685

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3869685,2022-12-18,17:00:00.000,International - FIFA World Cup,2022,Argentina,France,3,3,available,...,2024-12-16T10:21:13.710934,7,Final,Lusail Stadium,Szymon Marciniak,Lionel Sebastián Scaloni,Didier Deschamps,1.1.0,2,2
1,3869684,2022-12-17,17:00:00.000,International - FIFA World Cup,2022,Croatia,Morocco,2,1,available,...,2024-02-13T13:44:03.865296,7,3rd Place Final,Sheikh Khalifa International Stadium,Abdulrahman Ibrahim Al Jassim,Zlatko Dalić,Walid Regragui,1.1.0,2,2
2,3869552,2022-12-14,21:00:00.000,International - FIFA World Cup,2022,France,Morocco,2,0,available,...,2024-02-06T16:45:42.775801,6,Semi-finals,Al Bayt Stadium,César Arturo Ramos Palazuelos,Didier Deschamps,Hoalid Regragui,1.1.0,2,2
3,3869519,2022-12-13,21:00:00.000,International - FIFA World Cup,2022,Argentina,Croatia,3,0,available,...,2023-04-26T22:32:37.808359,6,Semi-finals,Lusail Stadium,Daniele Orsato,Lionel Sebastián Scaloni,Zlatko Dalić,1.1.0,2,2
4,3869486,2022-12-10,17:00:00.000,International - FIFA World Cup,2022,Morocco,Portugal,1,0,available,...,2024-02-06T16:39:05.200810,5,Quarter-finals,Al Thumama Stadium,Facundo Tello Figueroa,Hoalid Regragui,Fernando Manuel Fernandes da Costa Santos,1.1.0,2,2


In [141]:
# antes de passar para a validação de toda a informação do jogo, vou já deixar feita a estrutura da tabela de matches
# vou aproveitar para melhorar o formato da data e da season, para responder melhor depois na visualização

matches_df = matches[
    [
        'match_id',
        'match_date',
        'kick_off',
        'competition',
        'season',
        'home_team',
        'away_team',
        'home_score',
        'away_score',
        'match_status',
        'match_week',
        'competition_stage',
        'stadium',
        'referee',
        'home_managers',
        'away_managers'
    ]
].sort_values(
    by=['match_date', 'kick_off'],
    ascending=[False, False]
)

matches_df['match_date'] = pd.to_datetime(matches_df['match_date'])
matches_df['season'] = matches_df['season'].astype(str)

matches_df


# ESTA SERÁ A MINHA DIM_MATCHES NA ESTRUTURA STARSCHEMA

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,match_week,competition_stage,stadium,referee,home_managers,away_managers
0,3869685,2022-12-18,17:00:00.000,International - FIFA World Cup,2022,Argentina,France,3,3,available,7,Final,Lusail Stadium,Szymon Marciniak,Lionel Sebastián Scaloni,Didier Deschamps
1,3869684,2022-12-17,17:00:00.000,International - FIFA World Cup,2022,Croatia,Morocco,2,1,available,7,3rd Place Final,Sheikh Khalifa International Stadium,Abdulrahman Ibrahim Al Jassim,Zlatko Dalić,Walid Regragui
2,3869552,2022-12-14,21:00:00.000,International - FIFA World Cup,2022,France,Morocco,2,0,available,6,Semi-finals,Al Bayt Stadium,César Arturo Ramos Palazuelos,Didier Deschamps,Hoalid Regragui
3,3869519,2022-12-13,21:00:00.000,International - FIFA World Cup,2022,Argentina,Croatia,3,0,available,6,Semi-finals,Lusail Stadium,Daniele Orsato,Lionel Sebastián Scaloni,Zlatko Dalić
5,3869354,2022-12-10,21:00:00.000,International - FIFA World Cup,2022,England,France,1,2,available,5,Quarter-finals,Al Bayt Stadium,Wilton Pereira Sampaio,Gareth Southgate,Didier Deschamps
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,3857300,2022-11-22,10:00:00.000,International - FIFA World Cup,2022,Argentina,Saudi Arabia,1,2,available,1,Group Stage,Lusail Stadium,Slavko Vinčić,Lionel Sebastián Scaloni,Hervé Renard
61,3857282,2022-11-21,19:00:00.000,International - FIFA World Cup,2022,United States,Wales,1,1,available,1,Group Stage,Ahmad bin Ali Stadium,Abdulrahman Ibrahim Al Jassim,Gregg Berhalter,Robert Page
60,3857285,2022-11-21,16:00:00.000,International - FIFA World Cup,2022,Senegal,Netherlands,0,2,available,1,Group Stage,Al Thumama Stadium,Wilton Pereira Sampaio,Aliou Cissé,Louis van Gaal
62,3857271,2022-11-21,13:00:00.000,International - FIFA World Cup,2022,England,Iran,6,2,available,1,Group Stage,Sheikh Khalifa International Stadium,Raphael Claus,Gareth Southgate,Carlos Manuel Brito Leal Queiróz


****

<br> 

## Match Data Validation

Agora que já tenho o match_id escolhido, preciso de perceber se se confirma que tem a info pretendida (lineups, events, frames)

<br>

**LINEUPS**

In [142]:
# tipo de dados do dataset

lineups = sb.lineups(match_id=3869685)

type(lineups)

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


dict

In [143]:
# chaves da informação do dataset

lineups.keys()

dict_keys(['France', 'Argentina'])

In [144]:
# análise mais exploratória ao dataset lineups para cada uma das equipas presentes no jogo (info de colunas e linhas)

for team, df in lineups.items():
    print(f"\n{team}")
    print(df.shape)
    print(df.columns.tolist())


France
(24, 7)
['player_id', 'player_name', 'player_nickname', 'jersey_number', 'country', 'cards', 'positions']

Argentina
(26, 7)
['player_id', 'player_name', 'player_nickname', 'jersey_number', 'country', 'cards', 'positions']


In [145]:
# informação presente no dataset lineups para uma das equipas

lineups['Argentina'].head()

,player_id,player_name,player_nickname,jersey_number,country,cards,positions
0,2995,Ángel Fabián Di María Hernández,Ángel Di María,11,Argentina,[],"[{'position_id': 21, 'position': 'Left Wing', ..."
1,3090,Nicolás Hernán Otamendi,Nicolás Otamendi,19,Argentina,[],"[{'position_id': 5, 'position': 'Left Center B..."
2,5503,Lionel Andrés Messi Cuccittini,Lionel Messi,10,Argentina,[],"[{'position_id': 17, 'position': 'Right Wing',..."
3,5507,Nicolás Alejandro Tagliafico,Nicolás Tagliafico,3,Argentina,[],"[{'position_id': 6, 'position': 'Left Back', '..."
4,5743,Paulo Bruno Exequiel Dybala,Paulo Dybala,21,Argentina,[],"[{'position_id': 8, 'position': 'Left Wing Bac..."


**Considerações:**

- Existe uma coluna (positions) que precisaria de ser explodida porque tem bastante informação. No entanto, como isto será um dataset de lineups e talvez a informação da posição dos jogadores ser-me-á dada nos events
- Vou também ficar apenas com a posição 'mais' de cada jogador para simplificar o processo
- Retirar também a coluna cards porque precisa de ser explodida também e vou ter essa info nos events
- Passar a coluna main_position_id para número inteiro
- Atribuir números às posições para depois conseguir ordená-las na visualização

In [146]:
# coancatenar primeiro linesups das duas equipas para me dar um só dataset 
argentina_df = lineups['Argentina'].copy()
france_df = lineups['France'].copy()

players_df = pd.concat(
    [argentina_df, france_df],
    ignore_index=True
)


# obter posição principal do jogador
players_df['position'] = (
    players_df['positions']
    .apply(lambda x: x[0]['position'] if len(x) > 0 else None)
)

players_df['position_id'] = (
    players_df['positions']
    .apply(lambda x: x[0]['position_id'] if len(x) > 0 else None)
)


# renomear country para team
players_df = players_df.rename(
    columns={'country': 'team'}
)


# atribuir números às posições
position_order_map = {
    'Goalkeeper': 1,
    'Left Back': 2,
    'Left Wing Back': 3,
    'Right Back': 4,
    'Left Center Back': 5,
    'Right Center Back': 6,
    'Left Defensive Midfield': 7,
    'Center Defensive Midfield': 8,
    'Right Defensive Midfield': 9,
    'Left Center Midfield': 10,
    'Right Center Midfield': 11,
    'Center Attacking Midfield': 12,
    'Left Wing': 13,
    'Right Wing': 14,
    'Center Forward': 15,
    None: 99  # ou outro valor para posições em falta
}

players_df['position_order'] = players_df['position'].map(position_order_map)


# selecionar apenas as colunas relevantes
players_df = players_df[
    [
        'player_id',
        'player_name',
        'player_nickname',
        'jersey_number',
        'team',
        'position_id',
        'position',
        'position_order',        
    ]
]

# 'forçar' tipos das colunas
players_df['player_id'] = players_df['player_id'].astype('string')
players_df['jersey_number'] = players_df['jersey_number'].astype('string')
players_df['position_id'] = pd.to_numeric(players_df['position_id'], errors='coerce').astype('Int64')
players_df['position_order'] = pd.to_numeric(players_df['position_order'], errors='coerce').astype('Int64')

players_df.head(5)

,player_id,player_name,player_nickname,jersey_number,team,position_id,position,position_order
0,2995,Ángel Fabián Di María Hernández,Ángel Di María,11,Argentina,21,Left Wing,13
1,3090,Nicolás Hernán Otamendi,Nicolás Otamendi,19,Argentina,5,Left Center Back,5
2,5503,Lionel Andrés Messi Cuccittini,Lionel Messi,10,Argentina,17,Right Wing,14
3,5507,Nicolás Alejandro Tagliafico,Nicolás Tagliafico,3,Argentina,6,Left Back,2
4,5743,Paulo Bruno Exequiel Dybala,Paulo Dybala,21,Argentina,8,Left Wing Back,3


**Outras considerações:**

- Agora, e já pensando na parte da visualização, quero os logos das equipas, fotos dos jogadores, coordenadas da posição e se foram titulares ou não
- Apenas vou colocar de jogadores que tenham shirt_number associado
- Como são poucos jogadores e apenas duas equipas, vou diretamente ao fotmob buscar os links e posteriormente, farei um merge com players_df ficarei com a info que preciso

In [147]:
# fotos jogadores

player_images = {
    'Ángel Fabián Di María Hernández': 'https://images.fotmob.com/image_resources/playerimages/46509.png',
    'Nicolás Hernán Otamendi': 'https://images.fotmob.com/image_resources/playerimages/174321.png',
    'Lionel Andrés Messi Cuccittini': 'https://images.fotmob.com/image_resources/playerimages/30981.png',
    'Nicolás Alejandro Tagliafico': 'https://images.fotmob.com/image_resources/playerimages/195750.png',
    'Paulo Bruno Exequiel Dybala': 'https://images.fotmob.com/image_resources/playerimages/325916.png',
    'Damián Emiliano Martínez': 'https://images.fotmob.com/image_resources/playerimages/268375.png',
    'Germán Alejandro Pezzella': 'https://images.fotmob.com/image_resources/playerimages/186991.png',
    'Rodrigo Javier De Paul': 'https://images.fotmob.com/image_resources/playerimages/324578.png',
    'Lautaro Javier Martínez': 'https://images.fotmob.com/image_resources/playerimages/690230.png',
    'Leandro Daniel Paredes': 'https://images.fotmob.com/image_resources/playerimages/237606.png',
    'Marcos Javier Acuña': 'https://images.fotmob.com/image_resources/playerimages/561187.png',
    'Cristian Gabriel Romero': 'https://images.fotmob.com/image_resources/playerimages/789066.png',
    'Alexis Mac Allister': 'https://images.fotmob.com/image_resources/playerimages/831489.png',
    'Gonzalo Ariel Montiel': 'https://images.fotmob.com/image_resources/playerimages/687008.png',
    'Nahuel Molina Lucero': 'https://images.fotmob.com/image_resources/playerimages/726345.png',
    'Julián Álvarez': 'https://images.fotmob.com/image_resources/playerimages/974753.png',
    'Enzo Fernandez': 'https://images.fotmob.com/image_resources/playerimages/1137705.png',
    'Marcus Thuram': 'https://images.fotmob.com/image_resources/playerimages/621562.png',
    'Kylian Mbappé Lottin': 'https://images.fotmob.com/image_resources/playerimages/701154.png',
    'Adrien Rabiot': 'https://images.fotmob.com/image_resources/playerimages/352879.png',
    'Hugo Lloris': 'https://images.fotmob.com/image_resources/playerimages/26295.png',
    'Olivier Giroud': 'https://images.fotmob.com/image_resources/playerimages/46469.png',
    'Jules Koundé': 'https://images.fotmob.com/image_resources/playerimages/705450.png',
    'Ousmane Dembélé': 'https://images.fotmob.com/image_resources/playerimages/692984.png',
    'Raphaël Varane': 'https://images.fotmob.com/image_resources/playerimages/230982.png',
    'Antoine Griezmann': 'https://images.fotmob.com/image_resources/playerimages/184138.png', 
    'Theo Bernard François Hernández': 'https://images.fotmob.com/image_resources/playerimages/724371.png',
    'Kingsley Coman': 'https://images.fotmob.com/image_resources/playerimages/429265.png',
    'Dayotchanculle Upamecano': 'https://images.fotmob.com/image_resources/playerimages/658554.png',
    'Aurélien Djani Tchouaméni': 'https://images.fotmob.com/image_resources/playerimages/914458.png',
    'Ibrahima Konaté': 'https://images.fotmob.com/image_resources/playerimages/820140.png',
    'Youssouf Fofana': 'https://images.fotmob.com/image_resources/playerimages/917802.png',
    'Randal Kolo Muani': 'https://images.fotmob.com/image_resources/playerimages/823825.png',
    'Eduardo Camavinga': 'https://images.fotmob.com/image_resources/playerimages/1015185.png'

}

In [148]:
# logos equipas

team_logos = {
    'Argentina': 'https://images.fotmob.com/image_resources/logo/teamlogo/6706.png',
    'France': 'https://images.fotmob.com/image_resources/logo/teamlogo/6723.png'
}

In [149]:
# coordenadas das posições

position_coords = {

    # ARGENTINA (4x3x3)
    
    'Damián Emiliano Martínez': (6, 40),

    'Nahuel Molina Lucero': (18, 70),
    'Cristian Gabriel Romero': (18, 52),
    'Nicolás Hernán Otamendi': (18, 28),
    'Nicolás Alejandro Tagliafico': (18, 10),

    'Rodrigo Javier De Paul': (38, 54),
    'Enzo Fernandez': (32, 40),
    'Alexis Mac Allister': (38, 26),

    'Lionel Andrés Messi Cuccittini': (52, 62),
    'Julián Álvarez': (48, 40),
    'Ángel Fabián Di María Hernández': (52, 18),


    # FRANÇA (4x2x3x1)

    'Hugo Lloris': (114, 40),

    'Jules Koundé': (102, 10),
    'Raphaël Varane': (102, 28),
    'Dayotchanculle Upamecano': (102, 52),
    'Theo Bernard François Hernández': (102, 70),

    'Aurélien Djani Tchouaméni': (88, 26),
    'Adrien Rabiot': (88, 54),

    'Ousmane Dembélé': (76, 14),
    'Antoine Griezmann': (78, 40),
    'Kylian Mbappé Lottin': (76, 66),

    'Olivier Giroud': (68, 40)
}

In [150]:
# titulares (1 sim, 0 não)

starters = {
    'Ángel Fabián Di María Hernández',
    'Nicolás Hernán Otamendi',
    'Lionel Andrés Messi Cuccittini',
    'Nicolás Alejandro Tagliafico',
    'Damián Emiliano Martínez',
    'Rodrigo Javier De Paul',
    'Cristian Gabriel Romero',
    'Alexis Mac Allister',
    'Nahuel Molina Lucero',
    'Julián Álvarez',
    'Enzo Fernandez',
    'Kylian Mbappé Lottin',
    'Adrien Rabiot',
    'Hugo Lloris',
    'Olivier Giroud',
    'Jules Koundé',
    'Ousmane Dembélé',
    'Raphaël Varane',
    'Antoine Griezmann',
    'Theo Bernard François Hernández',
    'Dayotchanculle Upamecano',
    'Aurélien Djani Tchouaméni'
}

In [151]:
# adicionar as quatro colunas à tabela

players_df['player_photo'] = (players_df['player_name'].map(player_images))
players_df['team_logo'] = (players_df['team'].map(team_logos))

coords_df = pd.DataFrame.from_dict(
    position_coords,
    orient='index',
    columns=['x', 'y']
).reset_index()

coords_df.columns = ['player_name', 'x', 'y']

players_df = players_df.merge(
    coords_df,
    on='player_name',
    how='left'
)

players_df['starter'] = (
    players_df['player_name']
    .isin(starters)
    .astype(int)
)

In [152]:
# finalmente, uma última mudança, caso a coluna player_nickname seja None, vamos assumir o valor de player_name

players_df['player_nickname'] = (players_df['player_nickname'].fillna(players_df['player_name']))
players_df

# ESTA SERÁ A MINHA DIM_PLAYERS NA ESTRUTURA STARSCHEMA

,player_id,player_name,player_nickname,jersey_number,team,position_id,position,position_order,player_photo,team_logo,x,y,starter
0,2995,Ángel Fabián Di María Hernández,Ángel Di María,11,Argentina,21,Left Wing,13,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,52.0,18.0,1
1,3090,Nicolás Hernán Otamendi,Nicolás Otamendi,19,Argentina,5,Left Center Back,5,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,18.0,28.0,1
2,5503,Lionel Andrés Messi Cuccittini,Lionel Messi,10,Argentina,17,Right Wing,14,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,52.0,62.0,1
3,5507,Nicolás Alejandro Tagliafico,Nicolás Tagliafico,3,Argentina,6,Left Back,2,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,18.0,10.0,1
4,5743,Paulo Bruno Exequiel Dybala,Paulo Dybala,21,Argentina,8,Left Wing Back,3,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,NaN,NaN,0
5,6312,Franco Armani,Franco Armani,1,Argentina,<NA>,None,99,NaN,https://images.fotmob.com/image_resources/logo...,NaN,NaN,0
6,6377,Ángel Martín Correa,Ángel Correa,15,Argentina,<NA>,None,99,NaN,https://images.fotmob.com/image_resources/logo...,NaN,NaN,0
7,6694,Gerónimo Rulli,Gero Rulli,12,Argentina,<NA>,None,99,NaN,https://images.fotmob.com/image_resources/logo...,NaN,NaN,0
8,6909,Damián Emiliano Martínez,Emiliano Martínez,23,Argentina,1,Goalkeeper,1,https://images.fotmob.com/image_resources/play...,https://images.fotmob.com/image_resources/logo...,6.0,40.0,1
9,7006,Alejandro Darío Gómez,Papu Gómez,17,Argentina,<NA>,None,99,NaN,https://images.fotmob.com/image_resources/logo...,NaN,NaN,0


****

**EVENTS**

In [153]:
# vamos começar por perceber com que estrutura do dataset de eventos me vou deparar

events = sb.events(match_id=3869685)

events.shape

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(4407, 91)

In [154]:
# e também quais as colunas que existem

events.columns.tolist()
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4407 entries, 0 to 4406
Data columns (total 91 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   50_50                           8 non-null      object 
 1   bad_behaviour_card              2 non-null      object 
 2   ball_receipt_outcome            120 non-null    object 
 3   ball_recovery_offensive         1 non-null      object 
 4   ball_recovery_recovery_failure  13 non-null     object 
 5   block_deflection                1 non-null      object 
 6   block_offensive                 1 non-null      object 
 7   carry_end_location              940 non-null    object 
 8   clearance_aerial_won            8 non-null      object 
 9   clearance_body_part             45 non-null     object 
 10  clearance_head                  25 non-null     object 
 11  clearance_left_foot             8 non-null      object 
 12  clearance_other                 2 

<br>

**EVENTS DATASET EXPLORATION**

Agora que sei as colunas existentes em events, consigo perceber que já tenho ligações naturais aos dois dataframes anteriores:
    
    
- DIM_MATCHES - via match_id
- DIM_PLAYERS - via player_id (se não tiver player_id em todo o lado, como isto é apenas um jogo posso ligar por player_name)

O que me parece importante agora é ir construíndo um dataset de events com as colunas que creio que me vão interessar e ordena-las de forma a que o dataset faça sentido

Primeiras coisas que farei já porque sei que vou precisar:
- criar uma coluna de match_half (como este jogo teve prolongamento e penalties, vou criar 1, 2, 3, 4, 5) sendo que 3 e 4 é a primeira e segunda parte do prolongamento, e 5 são os penalties
- dividir location em location_x e location_y
- dividir pass_end_location em end_location_x e end_location_y
- vou assumir que os pass_outcome a null são complete e adicionar isso ao dataset

In [155]:
events_df = events.copy()

# criação de match_half
half_starts = (
    events_df[
        events_df['type'] == 'Half Start'
    ]
    .groupby('minute')['index']
    .min()
    .sort_values()
    .tolist()
)

events_df['match_half'] = 1

for i, start_index in enumerate(half_starts[1:], start=2):

    events_df.loc[
        events_df['index'] >= start_index,
        'match_half'
    ] = i
    

# coordenadas de origem do evento
events_df['location_x'] = (
    events_df['location']
    .apply(lambda x: x[0] if isinstance(x, list) else None)
)

events_df['location_y'] = (
    events_df['location']
    .apply(lambda x: x[1] if isinstance(x, list) else None)
)


# coordenadas de destino do passe
events_df['pass_end_x'] = (
    events_df['pass_end_location']
    .apply(lambda x: x[0] if isinstance(x, list) else None)
)

events_df['pass_end_y'] = (
    events_df['pass_end_location']
    .apply(lambda x: x[1] if isinstance(x, list) else None)
)


# substituir NULLs por Complete no outcome do passe
events_df['pass_outcome'] = events_df['pass_outcome'].fillna('Complete')


# forçar tipos de colunas
events_df['player_id'] = pd.to_numeric(events_df['player_id'], errors='coerce').astype('Int64')
events_df['team_id'] = pd.to_numeric(events_df['team_id'], errors='coerce').astype('Int64')


# visualização das principais colunas
events_df = events_df[
    [
        'match_id',
        'id',
        'related_events',
        'index',
        'player_id',
        'player',
        'position',
        'team_id',
        'team',
        'timestamp',
        'minute',
        'second',
        'match_half',
        'type',
        'off_camera',
        'duration',
        'location_x',
        'location_y',
        'under_pressure',
        'pass_aerial_won',
        'pass_angle',
        'pass_assisted_shot_id',
        'pass_body_part',
        'pass_cross',
        'pass_deflected',
        'pass_end_x',
        'pass_end_y',
        'pass_goal_assist',
        'pass_height',
        'pass_inswinging',
        'pass_length',
        'pass_outcome',
        'pass_outswinging',
        'pass_recipient',
        'pass_shot_assist',
        'pass_switch',
        'pass_technique',
        'pass_through_ball',
        'pass_type',
        'shot_outcome',
        'shot_statsbomb_xg'
    ]
].sort_values(
    by=['minute', 'second'],
    ascending=[True, True]
)

events_df.head()

# NESTE MOMENTO ESTA SERÁ A MINHA FCT_EVENTS NA ESTRUTURA STARSCHEMA

,match_id,id,related_events,index,player_id,player,position,team_id,team,timestamp,...,pass_outcome,pass_outswinging,pass_recipient,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,shot_outcome,shot_statsbomb_xg
0,3869685,0584ee21-e3dd-4d9f-95a0-5b5e84be25c3,NaN,1,<NA>,NaN,NaN,779,Argentina,00:00:00.000,...,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3869685,b32679f8-942e-4122-96a2-015caf75e628,NaN,2,<NA>,NaN,NaN,771,France,00:00:00.000,...,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3869685,954f6855-de22-46a2-8d09-6fe94eec2b9b,[6404a8e8-afaf-489d-b65e-173a237ffed5],3,<NA>,NaN,NaN,771,France,00:00:00.000,...,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3869685,6404a8e8-afaf-489d-b65e-173a237ffed5,[954f6855-de22-46a2-8d09-6fe94eec2b9b],4,<NA>,NaN,NaN,779,Argentina,00:00:00.000,...,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,3869685,f651a6c4-55e3-4e0f-a178-59414ba83d6a,[97b5dc82-547a-4f93-a632-a2a8daf5ac98],5,5487,Antoine Griezmann,Center Attacking Midfield,771,France,00:00:00.578,...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN


****

**FRAMES**

In [156]:
# vamos começar por perceber com que estrutura do dataset de frames me vou deparar

frames = sb.frames(match_id=3869685)

frames.shape

C:\Users\Administrador\anaconda3\lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(61451, 7)

In [157]:
# nem todos os eventos talvez possuam informação espacial (360).
# antes de explorar os freeze frames, importa perceber quantos eventos do jogo possuem contexto espacial associado.

total_events = events_df['id'].nunique()

events_with_frames = (
    events_df['id']
    .isin(frames['id'])
    .sum()
)

coverage = (
    events_with_frames / total_events
)

print(f"Total events: {total_events}")
print(f"Events with 360 data: {events_with_frames}")
print(f"Coverage: {coverage:.1%}")

Total events: 4407
Events with 360 data: 3683
Coverage: 83.6%


**Observações:**

- aproximadamente 84% dos eventos do jogo possuem informação espacial associada
- os restantes eventos não possuem freeze frame disponível no dataset StatsBomb 360
- a análise espacial será desenvolvida apenas para eventos com contexto 360 disponível

In [158]:
# perceber quais as colunas que existem
frames.columns.tolist()
frames.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61451 entries, 0 to 61450
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            61451 non-null  object
 1   visible_area  61451 non-null  object
 2   match_id      61451 non-null  int64 
 3   teammate      61451 non-null  bool  
 4   actor         61451 non-null  bool  
 5   keeper        61451 non-null  bool  
 6   location      61451 non-null  object
dtypes: bool(3), int64(1), object(3)
memory usage: 2.1+ MB


**Observações:**

- O campo 'id' vai ser a chave de ligação aos eventos
- A localização dos jogadores está na coluna 'location'
- Teammate, actor e keeper funcionam são informações importantes para eventuais criações de métricas mais à frente

In [159]:
# perceber quantos jogadores são capturados em média por cada freeze frame

frames.groupby('id').size().describe()

count    3683.000000
mean       16.685039
std         2.984872
min         1.000000
25%        15.000000
50%        17.000000
75%        19.000000
max        21.000000
dtype: float64

In [160]:
# validar se existe apenas um ator por evento

frames.groupby('id')['actor'].sum().value_counts()

1    3683
Name: actor, dtype: int64

In [161]:
# validar a presença de guarda-redes nos freeze frames

frames.groupby('id')['keeper'].sum().describe()

count    3683.000000
mean        0.289709
std         0.453689
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: keeper, dtype: float64

In [162]:
# validar a relação entre eventos e freeze frames

frames_events = (
    frames
    .merge(
        events[
            [
                'id',
                'type',
                'player',
                'team'
            ]
        ],
        on='id',
        how='left'
    )
)

frames_events

,id,visible_area,match_id,teammate,actor,keeper,location,type,player,team
0,f651a6c4-55e3-4e0f-a178-59414ba83d6a,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",3869685,True,False,False,"[39.21695730971832, 44.77861301106889]",Pass,Antoine Griezmann,France
1,f651a6c4-55e3-4e0f-a178-59414ba83d6a,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",3869685,True,False,False,"[39.2235613627218, 29.49684745718256]",Pass,Antoine Griezmann,France
2,f651a6c4-55e3-4e0f-a178-59414ba83d6a,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",3869685,True,False,False,"[42.57667276588051, 67.52374056870202]",Pass,Antoine Griezmann,France
3,f651a6c4-55e3-4e0f-a178-59414ba83d6a,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",3869685,True,False,False,"[46.549119819701936, 12.613254283667075]",Pass,Antoine Griezmann,France
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",3869685,True,False,False,"[47.79830626119247, 43.13070641887103]",Pass,Antoine Griezmann,France
...,...,...,...,...,...,...,...,...,...,...
61446,54e27ba9-b9ae-44a3-ac8d-fc91e76a1b91,"[6.10280272391214, 80.0, 37.1010455294984, 7.9...",3869685,False,False,True,"[119.03438903263786, 39.207858662781796]",Shot,Aurélien Djani Tchouaméni,France
61447,bcc6d4ae-f71c-4566-bf2d-c12f910eb526,"[6.0830318986702, 80.0, 35.2414826693926, 9.61...",3869685,True,True,False,"[108.0999984741211, 40.099998474121094]",Shot,Leandro Daniel Paredes,Argentina
61448,bcc6d4ae-f71c-4566-bf2d-c12f910eb526,"[6.0830318986702, 80.0, 35.2414826693926, 9.61...",3869685,False,False,True,"[119.86654053624902, 39.32380397475128]",Shot,Leandro Daniel Paredes,Argentina
61449,66eaf262-999f-4953-a924-f1b596de4dbf,"[69.5981279002227, 80.0, 59.425388472539, 10.8...",3869685,True,True,False,"[108.0999984741211, 40.099998474121094]",Shot,Gonzalo Ariel Montiel,Argentina


In [163]:
# perceber que tipos de eventos possuem freeze frame

frames_events[
    ['id', 'type']
].drop_duplicates()['type'].value_counts()

Pass              998
Ball Receipt*     994
Carry             845
Pressure          301
Ball Recovery     100
Duel               80
Dribble            43
Clearance          42
Block              40
Foul Committed     36
Shot               35
Foul Won           32
Miscontrol         30
Interception       26
Dispossessed       26
Dribbled Past      25
Goal Keeper        22
50/50               6
Offside             1
Shield              1
Name: type, dtype: int64

**Conclusões:**

- pass, ball receipt* e carry representam a maioria dos eventos com freeze frame
- o dataset StatsBomb 360 fornece contexto espacial para ações de construção e progressão
- existe informação suficiente para desenvolver métricas relacionadas com espaço, pressão e tomada de decisão

In [164]:
# para finalizar, quero deixar um dataset já pronto para frames
# é importante dividir a coluna de location em location_x e location_y
# e ainda perceber quantos vértices do polígono sao visiveis em visible_area 

frames_df = frames.copy()


# dividir coluna de location em x e y
frames_df['frame_location_x'] = (
    frames_df['location']
    .apply(lambda x: x[0] if isinstance(x, list) else None)
)

frames_df['frame_location_y'] = (
    frames_df['location']
    .apply(lambda x: x[1] if isinstance(x, list) else None)
)


# pontos visíveis na visible_area
frames_df['visible_area_points'] = (
    frames_df['visible_area']
    .apply(lambda x: len(x) // 2 if isinstance(x, list) else None)
)

frames_df = frames_df[
    [
        'id',
        'match_id',
        'teammate',
        'actor',
        'keeper',
        'frame_location_x',
        'frame_location_y',
        'visible_area',
        'visible_area_points'
    ]
]

frames_df.head()

# NESTE MOMENTO ESTA SERÁ A MINHA FCT_FRAMES NA ESTRUTURA STARSCHEMA

,id,match_id,teammate,actor,keeper,frame_location_x,frame_location_y,visible_area,visible_area_points
0,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,39.216957,44.778613,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5
1,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,39.223561,29.496847,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5
2,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,42.576673,67.523741,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5
3,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,46.549120,12.613254,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,47.798306,43.130706,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5


<br>

---
## Modelo de Dados Proposto
---

**DIM_MATCHES**

Primary Key - match_id

<br>

**DIM_PLAYERS**

Primary Key - player_id

<br>

**FCT_EVENTS**

Primary Key - id (event_id)

Foreign Keys - match_id, player_id, team_id

<br>

**FCT_FRAMES**

Foreign Key - id (event_id)

<br>
<br>

**DIM_COMPETITIONS**

Esta dimensão foi identificada durante a fase de exploração dos dados.

No entanto, não será utilizada na modelação final devido à ausência de uma chave de ligação natural com a tabela de jogos disponibilizada pelo dataset analisado.

Por esse motivo, a análise será desenvolvida utilizando as dimensões e fact tables diretamente relacionadas com o jogo selecionado.

<br>

**Exportação dos Datasets:**

Os datasets produzidos durante a fase de exploração e ETL serão utilizados no notebook seguinte para desenvolvimento de features e métricas.

In [165]:
import os

os.makedirs('data/processed', exist_ok=True)

matches_df.to_csv(
    'data/processed/matches.csv',
    index=False
)

players_df.to_csv(
    'data/processed/players.csv',
    index=False
)

events_df.to_csv(
    'data/processed/events.csv',
    index=False
)

frames_df.to_csv(
    'data/processed/frames.csv',
    index=False
)

print("Datasets exportados com sucesso.")

Datasets exportados com sucesso.


<br>

## Próximos Passos

Concluída a fase de exploração e preparação dos dados, a próxima etapa do projeto consistirá em:

- integrar Event Data e StatsBomb 360 Data
- desenvolver features espaciais ao nível do evento
- construir métricas personalizadas relacionadas com espaço, pressão e tomada de decisão
- preparar os datasets finais para consumo em Power BI

Os datasets tratados e produzidos nesta fase serão utilizados como base para o notebook de Feature Engineering.